In [14]:
import pyomo.environ as pyo
import time
from networkx.readwrite.gml import read_gml
import networkx as nx
import pickle
import os
from itertools import islice
from networkx.algorithms.simple_paths import shortest_simple_paths

eps = 0.0001

def services_to_features(services):
    return [demand['origin'] for service in services for demand in service],\
[demand['pmin'] for service in services for demand in service],\
[demand['pmax'] for service in services for demand in service],\
[demand['latency'] for service in services for demand in service],\
[sum(demand['resources'] for demand in service) for service in services]

def get_info_from_results(results, info_string):
    i = str(results).lower().find(info_string.lower()) + len(info_string)
    value = ''
    while str(results)[i] != '\n':
        value = value + str(results)[i]
        i += 1
    return value

graph_name = 'data/germany/germany.gml'
services_name = 'data/germany/g_03.pkl'

limit = 1

print('Contexte ok')

G = read_gml(graph_name)
G = nx.relabel.relabel_nodes(G, {node : int(node) for node in G.nodes()})
with open(services_name, 'rb') as f:
    services = pickle.load(f)
    
S, D, Sd = len(services), sum(len(elem) for elem in services), [len(elem) for elem in services]
E, A, arc_dic, arc_list = len(G.nodes), len(G.edges), {elem : i for i, elem in enumerate(G.edges)}, list(G.edges)
origins, Pdmin, Pdmax, latencies, resources = services_to_features(services)
alpha, beta = [G.edges[edge]['alpha'] for edge in G.edges], [G.edges[edge]['beta'] for edge in G.edges]

def partial_list_from_gen(gen, limit):
    new_list = []
    for i, elem in enumerate(gen):
        if i > limit:
            break
        new_list.append(elem)
    return new_list

shortest_paths = [[partial_list_from_gen(nx.algorithms.simple_paths.shortest_simple_paths(G, origins[d], target), limit) for target in G.nodes] for d in range(D)]
paths = [path[0] for l_path in shortest_paths for path in l_path]
P = len(paths)

print('Constantes ok')

model = pyo.ConcreteModel()

model.A = pyo.Set(initialize = range(A))
model.P = pyo.Set(initialize = range(P))
model.E = pyo.Set(initialize = range(E))
model.S = pyo.Set(initialize = range(S))
model.D = pyo.Set(initialize = range(D))

model.L = pyo.Var(model.P, model.A, within = pyo.NonNegativeReals)
model.Q = pyo.Var(model.P, within = pyo.NonNegativeReals)
model.q = pyo.Var(model.D, within = pyo.NonNegativeReals)
model.val_q = pyo.Constraint(model.D, rule = lambda model, d : pyo.inequality(Pdmin[d], model.q[d], Pdmax[d]))
model.y = pyo.Var(model.S, model.E, within = pyo.NonNegativeReals, bounds = (0,1))
model.l = pyo.Var(model.P, within = pyo.NonNegativeReals, bounds = (0,1))
model.z = pyo.Var(model.P, within = pyo.Binary)
model.nul = pyo.Var(range(1))

print('Variables ok')

model.obj = pyo.Objective(sense = pyo.maximize, rule = lambda model : sum(model.q.values()))

model.res = pyo.Constraint(model.E, rule = lambda model, e : sum(model.y[s, e]*resources[s] for s in model.S) <= G.nodes[e]['resources'])

def delta(c, a):
    return int(arc_list[a] in [(paths[c][i], paths[c][i+1]) for i in range(len(paths[c])-1)])

model.capa = pyo.Constraint(model.A, rule = lambda model, a : model.nul[0] + sum(delta(p, a)*model.Q[p] for p in model.P) <= G.edges[arc_list[a]]['capacity'])

p_to_d = [i for i in range(len(shortest_paths)) for j in range(len(shortest_paths[i]))]

def path_to_demand(p):
    return p_to_d[p]
    
model.lat = pyo.Constraint(model.P, model.A, rule = lambda model, p, a : model.L[p, a] <= latencies[path_to_demand(p)])

model.ind1 = pyo.Constraint(model.P, rule = lambda model, p : model.l[p] <= model.z[p])

model.ind2 = pyo.Constraint(model.P, rule = lambda model, p : model.z[p] <= 1+model.l[p]-eps)

model.res_reach = pyo.Constraint(model.S, rule = lambda model, s : sum(model.y[s, e] for e in model.E) >= 1)

def gamma(p, e):
    return int(paths[p][-1] == e)

pd = [0]+[sum(len(elem) for elem in shortest_paths[:i]) for i in range(1, len(shortest_paths)+1)]

model.serv_number = pyo.Param(model.D, 
                              initialize = {sum(len(elem) for elem in services[:i])+j : i for i, service in enumerate(services) for j, task in enumerate(service)}, 
                              mutable = False, 
                              within = pyo.NonNegativeIntegers)

def reach_serv(model, d, e):
    return model.y[model.serv_number[d], e] <= sum(gamma(p, e)*model.l[p] for p in range(pd[d], pd[d+1]))

model.reach_serv = pyo.Constraint(model.D, model.E, rule = reach_serv)

model.qual1 = pyo.Constraint(model.P, rule = lambda model, p : model.Q[p] <= model.q[path_to_demand(p)])

M = max(Pdmax)

model.qual2 = pyo.Constraint(model.P, rule = lambda model, p : model.Q[p] >= model.q[path_to_demand(p)] - M*(1-model.l[p]))

model.qual3 = pyo.Constraint(model.P, rule = lambda model, p : model.Q[p] <= M*model.l[p])

model.lat1 = pyo.Constraint(model.P, model.A, rule = lambda model, p, a : model.L[p, a] <= alpha[a]*sum(delta(pp, a)*model.Q[pp] for pp in model.P)+beta[a])

N = max(alpha) * sum(Pdmax) + max(beta)

model.lat2 = pyo.Constraint(model.P, model.A, rule = lambda model, p, a : model.L[p, a] >= alpha[a]*sum(delta(pp, a)*model.Q[pp] for pp in model.P)+beta[a] - N*(1-model.z[p]))

model.lat3 = pyo.Constraint(model.P, model.A, rule = lambda model, p, a : model.L[p, a] <= N*model.z[p])

print('solvage')

opt = pyo.SolverFactory('cplex')
options = {'time' : 3600, 'threads' : 1}
for key in options.keys():
    opt.options[key] = options[key]

resultat = opt.solve(model)

print('solvé')

res = ['Time: ', 'Lower bound: ', 'Upper bound: ', 'Termination condition: ']
results = {}
for elem in res:
    results[elem] = get_info_from_results(resultat, elem)
    
print(results)

Contexte ok
Constantes ok
Variables ok
solvage
solvé
{'Lower bound: ': '900.0', 'Termination condition: ': 'optimal', 'Upper bound: ': '900.0', 'Time: ': '357.94'}
